In [8]:
!git clone --branch Project_Update1 --single-branch \
  https://github.com/GustavoPCS/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project.git

%cd Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project

!git branch --show-current
!pip -q install transformers accelerate pandas numpy scikit-learn datasets pytest

Cloning into 'Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 52 (delta 8), reused 44 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 407.74 KiB | 7.55 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project
Project_Update1


In [7]:
# Download the original RAID input and recreate data/processed/dataset.csv
!mkdir -p data/raw
!curl -L -o data/raw/train_none.csv https://dataset.raid-bench.xyz/train_none.csv

!python -m src.build_dataset build \
  --csv data/raw/train_none.csv \
  --decoding greedy \
  --rep-penalty no

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  764M  100  764M    0     0  70.4M      0  0:00:10  0:00:10 --:--:-- 79.6M
label                         human  machine
domain    generator    split                
abstracts llama-chat   test      46       46
                       train    244      244
                       val       55       55
          mistral-chat test      64       64
                       train    342      342
                       val       77       77
news      llama-chat   test     134      134
                       train    617      617
                       val      133      133
          mistral-chat test     138      138
                       train    646      646
                       val      133      133
reddit    llama-chat   test      85       85
                       train    375      375
                       val       79       7

In [11]:
!python -m src.finetune_roberta \
  --data /content/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/data/processed/dataset.csv \
  --seeds 42 \
  --batch-size 16 \
  --epochs 3 \
  --out results/roberta_seed42

config.json: 100% 481/481 [00:00<00:00, 2.54MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 173kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 22.4MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 29.2MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 64.8MB/s]

model.safetensors: downloading bytes:  48% 237M/499M [00:01<00:00, 289MB/s, 19.5MB/s  ]
model.safetensors: reconstructing file:  54% 268M/499M [00:01<00:00, 236MB/s]
model.safetensors: downloading bytes:  64% 320M/499M [00:01<00:00, 235MB/s, 29.4MB/s  ]
model.safetensors: downloading bytes: 100% 335M/335M [00:01<00:00, 173MB/s, 30.5MB/s  ]
model.safetensors: reconstructing file: 100% 499M/499M [00:01<00:00, 257MB/s, 46.7MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 3511.50it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNE

In [12]:
import pandas as pd

display(pd.read_csv("results/roberta_seed42/metrics.csv"))
display(pd.read_csv("results/roberta_seed42/runtime.csv"))

,method,train_domain,train_generator,domain,generator,split,accuracy,precision_machine,recall_machine,f1_machine,...,auroc_lo,auroc_hi,accuracy_lo,accuracy_hi,tpr_lo,tpr_hi,fpr_lo,fpr_hi,macro_f1_in_domain,gap_macro_f1
0,roberta_base_seed42,abstracts,mistral-chat,abstracts,llama-chat,test,0.978261,0.978261,0.978261,0.978261,...,0.990662,1.000000,0.945040,1.000000,0.926829,1.000000,0.000000,0.072738,1.000000,0.021739
1,roberta_base_seed42,abstracts,mistral-chat,abstracts,mistral-chat,test,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
2,roberta_base_seed42,abstracts,mistral-chat,news,llama-chat,test,0.884328,0.912000,0.850746,0.880309,...,0.945449,0.977422,0.848142,0.917910,0.791036,0.906995,0.037587,0.131397,1.000000,0.115802
3,roberta_base_seed42,abstracts,mistral-chat,news,mistral-chat,test,0.829710,0.876033,0.768116,0.818533,...,0.903647,0.950270,0.787764,0.869892,0.695035,0.834562,0.063347,0.169017,1.000000,0.170938
4,roberta_base_seed42,abstracts,mistral-chat,reddit,llama-chat,test,0.776471,0.960784,0.576471,0.720588,...,0.873695,0.951632,0.720232,0.835383,0.466607,0.678171,0.000000,0.059524,1.000000,0.232843
5,roberta_base_seed42,abstracts,mistral-chat,reddit,mistral-chat,test,0.919192,0.966292,0.868687,0.914894,...,0.952630,0.992170,0.879212,0.955006,0.801887,0.931043,0.000000,0.066683,1.000000,0.081015
6,roberta_base_seed42,abstracts,mistral-chat,reviews,llama-chat,test,0.919540,0.876289,0.977011,0.923913,...,0.958352,0.996482,0.874285,0.955320,0.941843,1.000000,0.067416,0.219809,1.000000,0.080726
7,roberta_base_seed42,abstracts,mistral-chat,reviews,mistral-chat,test,0.902778,0.837209,1.000000,0.911392,...,0.943139,0.990975,0.861111,0.938686,1.000000,1.000000,0.122803,0.277790,1.000000,0.098150
8,roberta_base_seed42,news,mistral-chat,abstracts,llama-chat,test,0.597826,1.000000,0.195652,0.327273,...,0.929549,0.992556,0.500000,0.686901,0.093023,0.309102,0.000000,0.000000,0.989129,0.468904
9,roberta_base_seed42,news,mistral-chat,abstracts,mistral-chat,test,0.679688,1.000000,0.359375,0.528736,...,0.917384,0.980562,0.600000,0.754139,0.238788,0.475410,0.000000,0.000000,0.989129,0.346063


,method,seed,train_domain,train_generator,model,epochs,batch_size,max_length,n_train,train_seconds,inference_ms_per_1000_texts,peak_gpu_memory_gib_train,peak_gpu_memory_gib_inference
0,roberta_base_seed42,42,abstracts,mistral-chat,roberta-base,3,16,256,684,85.457556,15309.091700,3.933885,1.582640
1,roberta_base_seed42,42,news,mistral-chat,roberta-base,3,16,256,1292,184.964746,17213.197235,3.934007,1.585569
2,roberta_base_seed42,42,reviews,mistral-chat,roberta-base,3,16,256,846,120.222167,16209.753287,3.933885,1.584837
3,roberta_base_seed42,42,reddit,mistral-chat,roberta-base,3,16,256,906,127.415939,16301.198528,3.939866,1.589476


In [17]:
from google.colab import files
import shutil

# Create one ZIP file containing all RoBERTa outputs
shutil.make_archive(
    "roberta_seed42_results",
    "zip",
    root_dir="results/roberta_seed42"
)

files.download("roberta_seed42_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import pandas as pd
from pathlib import Path
from IPython.display import display

RESULT_DIR = Path("results/roberta_seed42")  # change to roberta_3seeds if needed
m = pd.read_csv(RESULT_DIR / "metrics.csv")

# Keep the most useful columns for the paper
display(m[[
    "method", "train_domain", "domain", "train_generator", "generator",
    "n", "macro_f1", "auroc", "fpr", "gap_macro_f1"
]].round(3))

,method,train_domain,domain,train_generator,generator,n,macro_f1,auroc,fpr,gap_macro_f1
0,roberta_base_seed42,abstracts,abstracts,mistral-chat,llama-chat,92,0.978,0.998,0.022,0.022
1,roberta_base_seed42,abstracts,abstracts,mistral-chat,mistral-chat,128,1.000,1.000,0.000,0.000
2,roberta_base_seed42,abstracts,news,mistral-chat,llama-chat,268,0.884,0.963,0.082,0.116
3,roberta_base_seed42,abstracts,news,mistral-chat,mistral-chat,276,0.829,0.927,0.109,0.171
4,roberta_base_seed42,abstracts,reddit,mistral-chat,llama-chat,170,0.767,0.915,0.024,0.233
5,roberta_base_seed42,abstracts,reddit,mistral-chat,mistral-chat,198,0.919,0.976,0.030,0.081
6,roberta_base_seed42,abstracts,reviews,mistral-chat,llama-chat,174,0.919,0.982,0.138,0.081
7,roberta_base_seed42,abstracts,reviews,mistral-chat,mistral-chat,216,0.902,0.970,0.194,0.098
8,roberta_base_seed42,news,abstracts,mistral-chat,llama-chat,92,0.520,0.969,0.000,0.469
9,roberta_base_seed42,news,abstracts,mistral-chat,mistral-chat,128,0.643,0.952,0.000,0.346


In [16]:
train_gen = "mistral-chat"
test_gen = "llama-chat"

same_domain = m[m["domain"] == m["train_domain"]].copy()

table1 = same_domain.pivot_table(
    index="train_domain",
    columns="generator",
    values=["macro_f1", "auroc", "fpr"],
    aggfunc="mean"
)

# Make readable column names
table1.columns = [
    f"{metric.replace('_', ' ').title()} ({generator})"
    for metric, generator in table1.columns
]

table1["Macro F1 Drop: Mistral → Llama"] = (
    table1[f"Macro F1 ({train_gen})"] -
    table1[f"Macro F1 ({test_gen})"]
)

table1 = table1.round(3)
display(table1)

print(table1.to_latex(
    caption="RoBERTa-base performance under same-domain generator shift. "
            "Models are trained on Mistral Chat (G1) and evaluated on Mistral Chat and Llama Chat (G2).",
    label="tab:roberta_generator_shift",
    float_format="%.3f"
))

,Auroc (llama-chat),Auroc (mistral-chat),Fpr (llama-chat),Fpr (mistral-chat),Macro F1 (llama-chat),Macro F1 (mistral-chat),Macro F1 Drop: Mistral → Llama
train_domain,,,,,,,
abstracts,0.998,1.0,0.022,0.0,0.978,1.000,0.022
news,1.000,1.0,0.000,0.0,0.985,0.989,0.004
reddit,0.998,1.0,0.000,0.0,0.994,1.000,0.006
reviews,0.999,1.0,0.000,0.0,0.730,1.000,0.270


\begin{table}
\caption{RoBERTa-base performance under same-domain generator shift. Models are trained on Mistral Chat (G1) and evaluated on Mistral Chat and Llama Chat (G2).}
\label{tab:roberta_generator_shift}
\begin{tabular}{lrrrrrrr}
\toprule
 & Auroc (llama-chat) & Auroc (mistral-chat) & Fpr (llama-chat) & Fpr (mistral-chat) & Macro F1 (llama-chat) & Macro F1 (mistral-chat) & Macro F1 Drop: Mistral → Llama \\
train_domain &  &  &  &  &  &  &  \\
\midrule
abstracts & 0.998 & 1.000 & 0.022 & 0.000 & 0.978 & 1.000 & 0.022 \\
news & 1.000 & 1.000 & 0.000 & 0.000 & 0.985 & 0.989 & 0.004 \\
reddit & 0.998 & 1.000 & 0.000 & 0.000 & 0.994 & 1.000 & 0.006 \\
reviews & 0.999 & 1.000 & 0.000 & 0.000 & 0.730 & 1.000 & 0.270 \\
\bottomrule
\end{tabular}
\end{table}

